# Best Diffusion Long-Form Panel on Downloads Songs

This notebook runs long-form diffusion inference on random songs from `Downloads/` using a curated checkpoint panel built from the strongest saved diffusion runs.

Focus:
- maximum style shift
- minimum compounded static / warble
- same random song plan across all selected checkpoints

Default curated panel:
- `run_d002_best`
- `run_d002_epoch_006`
- `reset_best`
- `reset_epoch_005`

The defaults are intentionally more style-forward than the safety-first notebook, but still keep frequent re-anchoring and lighter mel correction to reduce long-form collapse.


In [ ]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'dggr').exists() and (path / 'lab 4').exists() and (path / 'lab 3.1').exists():
            return path
    raise RuntimeError('Could not resolve repo root from current working directory.')

REPO = find_repo_root(Path.cwd().resolve())
SCRIPTS = REPO / 'lab 3.1' / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import diffusion_longform_compare as dlc
importlib.reload(dlc)
REPO

In [ ]:
cfg = dlc.make_style_shift_stable_config(
    downloads_dir=Path.home() / 'Downloads',
    n_songs=3,
    targets_per_song=2,
    source_seconds=45.0,
    seed=328,
)

RUN_ALL = False

ctx = dlc.ddb.resolve_inference_context(
    dlc.ddb.DiffusionDownloadsBatchConfig(run_dir=cfg.run_dir, cache_dir=cfg.cache_dir)
)
print('Resolved run dir:    ', ctx['run_dir'])
print('Resolved cache dir:  ', ctx['cache_dir'])
print('Planned output root: ', cfg.output_root / cfg.tag)
print('Style-forward stable defaults loaded.')

In [ ]:
checkpoint_panel = dlc.resolve_curated_best_diffusion_panel()
display(pd.DataFrame([
    {
        'label': row['label'],
        'path': str(row['path']),
        'note': row.get('note', ''),
    }
    for row in checkpoint_panel
]))
print('Total checkpoints:', len(checkpoint_panel))

In [ ]:
jobs = dlc.plan_longform_jobs(cfg)
display(pd.DataFrame(jobs))
print('Total planned long-form jobs:', len(jobs))
print('Total runs across checkpoints:', len(jobs) * len(checkpoint_panel))

In [ ]:
summary = None
if RUN_ALL:
    summary = dlc.run_longform_compare(cfg, checkpoint_panel)
    print(json.dumps(summary, indent=2, default=str))
else:
    print('Set RUN_ALL = True to launch the curated best-diffusion long-form batch.')

In [ ]:
summary_path = cfg.output_root / cfg.tag / 'summary.json'
manifest_path = cfg.output_root / cfg.tag / 'manifest.csv'
if summary_path.exists():
    print(summary_path)
    print(manifest_path)
    display(pd.read_csv(manifest_path))
else:
    print('No outputs yet for this tag.')